# REST interface for DB Service

The DB Service REST interface provides direct HTTP access to DB Service APIs, making it easier to connect to a running service and perform client operations using cURL.

Use this to run queries, manage tables, and interact with DB Service over REST.

## Managing Tables
Use these calls to define and inspect table schemas in DB Service.

In [1]:
%%bash
# List tables (empty to begin with)
curl -s -X GET "http://localhost:8080/api/v0/tables" -H "Accept: application/json" | jq

[]


In [2]:
%%bash
# Create partitioned table ('fxquote')
curl -s -X POST "http://localhost:8080/api/v0/tables/fxquote" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{
    "type": "partitioned",
    "prtnCol": "ts",
    "sortColsDisk": ["sym"],
    "sortColsOrd": ["sym"],
    "columns": [
      {"name": "trddate", "type": "date"},
      {"name": "ts", "type": "timestamp"},
      {"name": "sym", "type": "symbol", "attrMem": "grouped", "attrDisk": "parted", "attrOrd": "parted"},
      {"name": "bid", "type": "float"},
      {"name": "ask", "type": "float"}
    ]
  }' | jq

{
  "name": "b19f626d-f966-be0f-ac6f-a292080a534d",
  "pipeline": "",
  "database": "db",
  "updtype": "schemaChange",
  "status": "completed",
  "details": [],
  "tbls": [],
  "dates": [],
  "progress": {
    "cmdCurrent": "",
    "cmdIndex": 0,
    "cmdTotal": 0,
    "subCurrent": "",
    "subIndex": 0,
    "subTotal": 0
  },
  "error": "",
  "warnings": [],
  "updated": "2026-04-27T09:56:28.433020406"
}


In [3]:
%%bash
# List tables ('fxquote' table returned)
curl -s -X GET "http://localhost:8080/api/v0/tables" -H "Accept: application/json" | jq

[
  "fxquote"
]


In [4]:
%%bash
# Describe the 'fxquote' table
curl -s -X GET "http://localhost:8080/api/v0/tables/fxquote" -H "Accept: application/json" | jq

{
  "type": "partitioned",
  "prtnCol": "ts",
  "sortColsDisk": [
    "sym"
  ],
  "sortColsOrd": [
    "sym"
  ],
  "columns": [
    {
      "name": "trddate",
      "type": "date"
    },
    {
      "name": "ts",
      "type": "timestamp"
    },
    {
      "name": "sym",
      "type": "symbol",
      "attrMem": "grouped",
      "attrDisk": "parted",
      "attrOrd": "parted"
    },
    {
      "name": "bid",
      "type": "float"
    },
    {
      "name": "ask",
      "type": "float"
    }
  ],
  "name": "fxquote"
}


## Importing Data
DB Service supports both `file-based` and `in-memory` ingest. Any file you want to import must first be copied into the DB Service `imports` staging directory, for example: `~/.kx/db-service/data/imports/`

### Import CSV

In [ ]:
%%bash
# Import a CSV file into the existing 'fxquote' table
curl -s -X POST "http://localhost:8080/api/v0/imports/files" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"table": "fxquote", "path": "fxquote.csv.gz"}' | jq

{
  "name": "eded5623-144a-3923-99df-509628caaeba",
  "pipeline": "",
  "database": "db",
  "updtype": "ingest",
  "status": "pending",
  "details": [],
  "tbls": [],
  "dates": [],
  "progress": {
    "cmdCurrent": "",
    "cmdIndex": null,
    "cmdTotal": null,
    "subCurrent": "",
    "subIndex": null,
    "subTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-04-27T09:56:40.550000350"
}


In [6]:
%%bash
# Check the status of the above import job
curl -X GET "http://localhost:8080/api/v0/imports/NAME" -H "Accept: application/json" | jq

# Note: Replace `NAME` above, with the actual "name" returned from the previous import job.

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   447  100   447    0     0  17812      0 --:--:-- --:--:-- --:--:-- 17880


{
  "name": "eded5623-144a-3923-99df-509628caaeba",
  "pipeline": "",
  "database": "db",
  "updtype": "ingest",
  "status": "completed",
  "details": {
    "preprocessDone": "2026-04-27T09:56:41.468205843",
    "subsessions": [
      ""
    ],
    "dates": [
      "2026-03-02"
    ],
    "tables": [
      "fxquote"
    ]
  },
  "tbls": [
    "fxquote"
  ],
  "dates": [
    "2026-03-02"
  ],
  "progress": {
    "cmdCurrent": "",
    "cmdIndex": 1,
    "cmdTotal": 1,
    "subCurrent": "",
    "subIndex": 0,
    "subTotal": 0
  },
  "error": "",
  "warnings": [],
  "updated": "2026-04-27T09:56:41.748385863"
}


In [ ]:
%%bash
# Import a parquet file into the existing 'fxquote' table
curl -s -X POST "http://localhost:8080/api/v0/imports/files" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"table": "fxquote", "path": "fxquote.parquet"}' | jq

{
  "name": "0cc9a868-1337-4177-289f-b2b4a0bb4780",
  "pipeline": "",
  "database": "db",
  "updtype": "ingest",
  "status": "pending",
  "details": [],
  "tbls": [],
  "dates": [],
  "progress": {
    "cmdCurrent": "",
    "cmdIndex": null,
    "cmdTotal": null,
    "subCurrent": "",
    "subIndex": null,
    "subTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-04-27T09:57:08.077211128"
}


In [ ]:
%%bash
# Import a CSV file and create the 'instruments' table automatically if it does not exist
curl -s -X POST "http://localhost:8080/api/v0/imports/files" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"table": "instruments", "path": "instruments.csv", "createTable": "1"}' | jq

{
  "name": "d37ada94-b74e-8adf-ecc4-91a35e279224",
  "pipeline": "",
  "database": "db",
  "updtype": "ingest",
  "status": "pending",
  "details": [],
  "tbls": [],
  "dates": [],
  "progress": {
    "cmdCurrent": "",
    "cmdIndex": null,
    "cmdTotal": null,
    "subCurrent": "",
    "subIndex": null,
    "subTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-04-27T09:57:09.590976434"
}


### Import JSON
Users can import data directly without file staging.

In [ ]:
%%bash
# Objects payload imported to 'instruments' table
curl -s -X POST "http://localhost:8080/api/v0/imports/data" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{
    "table": "instruments",
    "data": [
      {"instrumentid": 77, "sym": "USDBRL", "category": "EM", "decimals": 4, "pipdecimals": 4},
      {"instrumentid": 78, "sym": "USDKRW", "category": "EM", "decimals": 2, "pipdecimals": 2}
    ],
    "insert_as": "objects"
  }' | jq

{
  "name": "fed03e37-91d0-66e3-b60c-ffc329f6dd53",
  "pipeline": "",
  "database": "db",
  "updtype": "ingest",
  "status": "pending",
  "details": [],
  "tbls": [],
  "dates": [],
  "progress": {
    "cmdCurrent": "",
    "cmdIndex": null,
    "cmdTotal": null,
    "subCurrent": "",
    "subIndex": null,
    "subTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-04-27T09:57:13.544409083"
}


In [ ]:
%%bash
# Rows payload imported to the 'fxquote' table
curl -s -X POST "http://localhost:8080/api/v0/imports/data" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{
    "table": "fxquote",
    "data": [
      ["2026-01-21", "2026-01-21T10:00:00.000", "EURUSD", 901.2, 901.3],
      ["2026-01-21", "2026-01-21T10:00:00.000", "EURUSD", 901.2, 901.3]
    ],
    "columnNames": ["trddate", "ts", "sym", "bid", "ask"],
    "insert_as": "rows"
  }' | jq

# Note: for rows payload, columnNames are required.

{
  "name": "4dbd6865-1a5e-016f-1eb8-9d5038d7039f",
  "pipeline": "",
  "database": "db",
  "updtype": "ingest",
  "status": "pending",
  "details": [],
  "tbls": [],
  "dates": [],
  "progress": {
    "cmdCurrent": "",
    "cmdIndex": null,
    "cmdTotal": null,
    "subCurrent": "",
    "subIndex": null,
    "subTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-04-27T09:57:14.920277599"
}


## Querying Tables
Run structured, SQL, or q queries against DB Service.

In [ ]:
%%bash
# Structured query
curl -s -X POST "http://localhost:8080/api/v0/query/simple" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"table": "fxquote", "startTS": "2026.03.02D00:00:00", "endTS": "2026.03.03D00:00:00", "limit": 2}' | jq

{
  "header": {
    "corr": "56c30964-7b66-4f3c-8a88-0efa0771b7d4",
    "logCorr": "56c30964-7b66-4f3c-8a88-0efa0771b7d4",
    "version": 1,
    "rcvTS": "2026-04-27T09:57:16.775000000",
    "http": "json",
    "api": ".query.simple",
    "agg": ":172.20.0.3:5060",
    "refVintage": -9223372036854775807,
    "rc": 0,
    "ac": 0,
    "ai": "",
    "limitApplied": true
  },
  "payload": [
    {
      "trddate": "2026-03-02",
      "ts": "2026-03-02T00:00:00.000000000",
      "sym": "AUDUSD",
      "bid": 0.67091,
      "ask": 0.67094
    },
    {
      "trddate": "2026-03-02",
      "ts": "2026-03-02T00:00:06.000000000",
      "sym": "AUDUSD",
      "bid": 0.67095,
      "ask": 0.67097
    }
  ]
}


In [ ]:
%%bash
# SQL query
curl -s -X POST "http://localhost:8080/api/v0/query/sql" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"query": "SELECT * FROM instruments WHERE category LIKE '\''EM'\''"}' | jq

{
  "header": {
    "corr": "0570cbd1-23ce-4868-a797-e15690819779",
    "logCorr": "0570cbd1-23ce-4868-a797-e15690819779",
    "version": 1,
    "rcvTS": "2026-04-27T09:57:17.526000000",
    "http": "json",
    "api": ".query.sql",
    "agg": ":172.20.0.3:5060",
    "refVintage": -9223372036854775807,
    "rc": 0,
    "ac": 0,
    "ai": ""
  },
  "payload": [
    {
      "instrumentid": 14,
      "sym": "CHFZAR",
      "category": "EM",
      "decimals": 5,
      "pipdecimals": 4
    },
    {
      "instrumentid": 29,
      "sym": "EURTRY",
      "category": "EM",
      "decimals": 5,
      "pipdecimals": 4
    },
    {
      "instrumentid": 31,
      "sym": "EURZAR",
      "category": "EM",
      "decimals": 5,
      "pipdecimals": 4
    },
    {
      "instrumentid": 41,
      "sym": "GBPZAR",
      "category": "EM",
      "decimals": 5,
      "pipdecimals": 4
    },
    {
      "instrumentid": 61,
      "sym": "USDCNH",
      "category": "EM",
      "decimals": 5,
      "pipdecimals

In [ ]:
%%bash
# QSQL query
curl -s -X POST "http://localhost:8080/api/v0/query/q" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"query": "select o:first bid,h:max bid,l:min bid,c:last bid by trddate,sym from fxquote"}' | jq

{
  "header": {
    "corr": "740fb465-208b-4be1-acc9-3424136cdcd5",
    "logCorr": "740fb465-208b-4be1-acc9-3424136cdcd5",
    "version": 1,
    "rcvTS": "2026-04-27T09:57:18.238000000",
    "http": "json",
    "api": ".query.q",
    "agg": ":172.20.0.3:5060",
    "refVintage": -9223372036854775807,
    "rc": 0,
    "ac": 0,
    "ai": ""
  },
  "payload": [
    {
      "trddate": "2026-01-21",
      "sym": "EURUSD",
      "o": 901.2,
      "h": 901.2,
      "l": 901.2,
      "c": 901.2
    },
    {
      "trddate": "2026-03-02",
      "sym": "AUDUSD",
      "o": 0.67091,
      "h": 0.67469,
      "l": 0.67065,
      "c": 0.67307
    },
    {
      "trddate": "2026-03-02",
      "sym": "EURUSD",
      "o": 1.16397,
      "h": 1.1768,
      "l": 1.16325,
      "c": 1.17268
    },
    {
      "trddate": "2026-03-02",
      "sym": "GBPUSD",
      "o": 1.3419,
      "h": 1.34913,
      "l": 1.34101,
      "c": 1.34404
    },
    {
      "trddate": "2026-03-02",
      "sym": "USDCAD",
      

**Return format:** `return_as` may be `json`, `pandas`, or `pykx`. If omitted, it defaults to `json`.

## Deleting tables

In [15]:
%%bash
# List tables (expected: 'fxquote' and 'instruments')
curl -s -X GET "http://localhost:8080/api/v0/tables" -H "Accept: application/json" | jq

[
  "fxquote",
  "instruments"
]


In [ ]:
%%bash
# Drop the 'instruments' table
curl -s -X DELETE "http://localhost:8080/api/v0/tables/instruments" -H "Accept: application/json" | jq

{
  "name": "9dfd60d2-1b48-cb67-1830-ef384eab9404",
  "pipeline": "",
  "database": "db",
  "updtype": "schemaChange",
  "status": "completed",
  "details": [],
  "tbls": [],
  "dates": [],
  "progress": {
    "cmdCurrent": "",
    "cmdIndex": 0,
    "cmdTotal": 0,
    "subCurrent": "",
    "subIndex": 0,
    "subTotal": 0
  },
  "error": "",
  "warnings": [],
  "updated": "2026-04-27T09:57:51.792059752"
}


In [17]:
%%bash
# List tables (expected: instruments is gone)
curl -s -X GET "http://localhost:8080/api/v0/tables" -H "Accept: application/json" | jq

[
  "fxquote"
]
